# HADDOCK Energy Analysis - HADDOCK_results

Analysis of Electrostatic, Van der Waals, Desolvation, and Total energies across HADDOCK runs in HADDOCK_results directory.

Note: Non-bonded energy (`Enb`) is excluded because it already equals Electrostatic + Van der Waals — including it in the total would double-count those terms. `Total_Energy = Eelec + Evdw + Edesolv`.


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from scipy import stats
from itertools import combinations

plt.style.use('default')
plt.rcParams['figure.figsize'] = (20, 8)
plt.rcParams['font.size'] = 12

In [ ]:
def read_haddock_energies(stat_file):
    """Read energy data from HADDOCK statistics file"""
    energy_data = []

    with open(stat_file, 'r') as f:
        lines = f.readlines()

        for line in lines[1:]:  # Skip header
            line = line.strip()
            if line and not line.startswith('#'):
                parts = line.split()
                if len(parts) >= 23:
                    try:
                        data = {
                            'Structure': parts[0],
                            'HADDOCK_Score': float(parts[1]),
                            'Evdw': float(parts[6]),          # Van der Waals energy
                            'Eelec': float(parts[7]),         # Electrostatic energy
                            'Edesolv': float(parts[22])       # Desolvation energy
                        }

                        # Total energy = Eelec + Evdw + Edesolv (3 components).
                        # Enb is omitted because Enb = Eelec + Evdw already (would double-count).
                        data['Total_Energy'] = data['Evdw'] + data['Eelec'] + data['Edesolv']

                        # Only include reasonable energies
                        if data['HADDOCK_Score'] < 1000:
                            energy_data.append(data)

                    except (ValueError, IndexError):
                        continue

    return energy_data

In [ ]:
# Define HADDOCK runs in HADDOCK_results directory
haddock_runs = {
    '672620-original_boltz2': 'Original Boltz2',
    '669247-4base_mismatch_boltz2': '4Base Mismatch Boltz2', 
    '668631-terminal_fold2': 'Terminal Fold2',
    '669245-scrambled_boltz2': 'Scrambled Boltz2'
}

# Load all data
all_data = []

print("📁 Loading HADDOCK data from HADDOCK_results...")
successful_runs = []

for folder, run_name in haddock_runs.items():
    stat_file = f'HADDOCK_results/{folder}/structures/it1/water/structures_air-sorted.stat'
    
    if os.path.exists(stat_file):
        energy_data = read_haddock_energies(stat_file)
        if len(energy_data) > 0:
            print(f"✅ {run_name}: {len(energy_data)} structures")
            successful_runs.append(run_name)
            
            for data in energy_data:
                data['Run_Name'] = run_name
                data['Run_ID'] = folder
                all_data.append(data)
        else:
            print(f"⚠️ {run_name}: No valid energy data found")
    else:
        print(f"⚠️ File not found: {stat_file}")

# Create DataFrame
df = pd.DataFrame(all_data)

# Show structure counts per run
print("\n📋 Structure counts:")
structure_counts = df.groupby('Run_Name').size().sort_values(ascending=False)
display(structure_counts.to_frame('Count'))

In [ ]:
# Define the 4 energy types for box plots (Enb excluded — see note in cell 0)
energy_types = {
    'Eelec': 'Electrostatic Energy',
    'Evdw': 'Van der Waals Energy',
    'Edesolv': 'Desolvation Energy',
    'Total_Energy': 'Total Energy (Sum of All 3)'
}

# Create 4 box plots in a single figure
fig, axes = plt.subplots(1, 4, figsize=(20, 6))

# Color scheme matching the PDB structure screenshot
# Terminal=purple, Original=red, 4Base Mismatch=orange, Scrambled=green
colors = ['#9370DB', '#B22222', '#FF6347', '#228B22']  # Purple, Burgundy, Orange-Red, Forest Green

# NEW ORDER: Terminal, Original, 4Base Mismatch, Scrambled
run_order = ['Terminal Fold2', 'Original Boltz2', '4Base Mismatch Boltz2', 'Scrambled Boltz2']
category_labels = ['Terminal', 'Original', '4 Base\nMismatch', 'Scrambled']

for idx, (energy_col, energy_title) in enumerate(energy_types.items()):
    ax = axes[idx]

    # Prepare data for box plot
    box_data = []
    box_colors = []

    for i, run in enumerate(run_order):
        run_data = df[df['Run_Name'] == run][energy_col]
        if len(run_data) > 0:
            box_data.append(run_data)
            box_colors.append(colors[i])

    # Create box plot with category labels
    bp = ax.boxplot(box_data,
                   labels=category_labels,
                   patch_artist=True,
                   boxprops=dict(linewidth=1.5),
                   whiskerprops=dict(linewidth=1.5),
                   capprops=dict(linewidth=1.5),
                   medianprops=dict(linewidth=2, color='black'),
                   flierprops=dict(marker='o', markersize=3, alpha=0.6))

    # Color the boxes
    for patch, color in zip(bp['boxes'], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    # Add scatter points
    for i, run in enumerate(run_order, 1):
        run_data = df[df['Run_Name'] == run][energy_col]
        if len(run_data) > 0:
            y = run_data.values
            x = np.random.normal(i, 0.04, size=len(y))
            ax.scatter(x, y, alpha=0.3, s=8, color='black')

    # Customize plot
    ax.set_title(f'{energy_title}', fontsize=14, fontweight='bold', pad=20)
    ax.set_ylabel('Energy (kcal/mol)', fontsize=12)
    ax.grid(axis='y', alpha=0.3, linestyle='--')

    # NO rotation for x-axis labels, smaller font to prevent overlap
    ax.tick_params(axis='x', rotation=0, labelsize=10)

    # Add mean and min values together with better positioning
    for i, run in enumerate(run_order, 1):
        run_data = df[df['Run_Name'] == run][energy_col]
        if len(run_data) > 0:
            mean_val = run_data.mean()
            min_val = run_data.min()
            y_range = ax.get_ylim()[1] - ax.get_ylim()[0]
            y_pos = ax.get_ylim()[1] - y_range * 0.15  # Even higher to avoid overlap

            # Use simple, clean notation that definitely renders
            text_content = f'$E_{{\\mu}}={mean_val:.0f}$\n$E_{{min}}={min_val:.0f}$'

            ax.text(i, y_pos, text_content,
                   ha='center', va='center', fontweight='bold', fontsize=9,
                   bbox=dict(boxstyle='round,pad=0.5', facecolor='white',
                           alpha=0.95, edgecolor='gray', linewidth=1))

plt.suptitle('HADDOCK Energy Analysis: 4 Energy Components Across 4 Runs',
             fontsize=16, fontweight='bold', y=0.98)

plt.tight_layout()
plt.subplots_adjust(bottom=0.2)  # Even more space at bottom for two-line labels

# Save in both PNG and PDF formats
plt.savefig('HADDOCK_results_4_Energy_BoxPlots.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('HADDOCK_results_4_Energy_BoxPlots.pdf', bbox_inches='tight', facecolor='white')
plt.show()

print("📊 Box plots generated with colors matching PDB structure screenshot!")
print("💾 Saved as both PNG and PDF formats")
print("🎨 Colors: Terminal=Purple, Original=Red, 4Base Mismatch=Orange, Scrambled=Green")

In [ ]:
# Statistical Analysis - P-value calculations between all pairs of runs
print("📈 Statistical Analysis: P-value calculations between runs")
print("=" * 70)

# Create a comprehensive p-value DataFrame
p_value_results = []

# Get all pairs of runs
run_pairs = list(combinations(run_order, 2))
print(f"Performing {len(run_pairs)} pairwise comparisons per energy type...")

for energy_col, energy_title in energy_types.items():
    print(f"\n🎯 {energy_title}:")
    print("-" * 50)
    
    for run1, run2 in run_pairs:
        data1 = df[df['Run_Name'] == run1][energy_col]
        data2 = df[df['Run_Name'] == run2][energy_col]
        
        if len(data1) > 0 and len(data2) > 0:
            # Perform t-test
            t_stat, p_val = stats.ttest_ind(data1, data2)
            
            # Store results
            p_value_results.append({
                'Energy_Type': energy_title,
                'Run_1': run1,
                'Run_2': run2,
                'P_Value': p_val,
                'Significant': 'Yes' if p_val < 0.05 else 'No',
                'Mean_1': data1.mean(),
                'Mean_2': data2.mean(),
                'Mean_Difference': abs(data1.mean() - data2.mean())
            })
            
            significance = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
            print(f"  {run1} vs {run2}: p = {p_val:.2e} {significance}")

# Create comprehensive p-value DataFrame
p_value_df = pd.DataFrame(p_value_results)

print(f"\n📊 Complete P-value Results:")
display(p_value_df.round(30))

In [ ]:
# Create summary statistics table
print("📊 Summary Statistics for All Energy Types")
print("=" * 60)

summary_stats = []

for energy_col, energy_title in energy_types.items():
    for run in run_order:
        run_data = df[df['Run_Name'] == run][energy_col]
        if len(run_data) > 0:
            summary_stats.append({
                'Energy_Type': energy_title,
                'Run_Name': run,
                'Count': len(run_data),
                'Mean': run_data.mean(),
                'Std': run_data.std(),
                'Min': run_data.min(),
                'Max': run_data.max(),
                'Median': run_data.median()
            })

summary_df = pd.DataFrame(summary_stats)

# Show ranking by energy type (best = most negative)
print("\n🏆 ENERGY RANKINGS (Most Negative = Best Binding):")
print("=" * 60)

for energy_col, energy_title in energy_types.items():
    print(f"\n🎯 {energy_title}:")
    ranking_data = summary_df[summary_df['Energy_Type'] == energy_title].copy()
    ranking_data = ranking_data.sort_values('Mean')  # Most negative first
    
    for i, row in enumerate(ranking_data.itertuples(), 1):
        print(f"  {i}. {row.Run_Name:<25}: {row.Mean:8.1f} kcal/mol (n={row.Count})")

# Display full summary table
print("\n📊 Complete Summary Statistics:")
display(summary_df.round(2))

# Save results
summary_df.to_csv('HADDOCK_results_Energy_Summary_Stats.csv', index=False)
p_value_df.to_csv('HADDOCK_results_Energy_PValues.csv', index=False)

# Save significant results only
significant_results = p_value_df[p_value_df['Significant'] == 'Yes'].copy()
significant_results = significant_results.sort_values('P_Value')
significant_results.to_csv('HADDOCK_results_Significant_PValues.csv', index=False)

print("\n💾 Files Saved:")
print("  📊 HADDOCK_results_4_Energy_BoxPlots.png")
print("  📊 HADDOCK_results_Energy_Summary_Stats.csv")
print("  📊 HADDOCK_results_Energy_PValues.csv")
print("  📊 HADDOCK_results_Significant_PValues.csv (p < 0.05 only)")
print(f"\n✨ Analysis Complete!")
print(f"📊 Total structures analyzed: {len(df)}")
print(f"📊 Significant comparisons: {len(significant_results)} out of {len(p_value_df)}")

In [ ]:
# Quick visualization of energy differences from Original
print("🎯 ENERGY DIFFERENCES FROM ORIGINAL BOLTZ2")
print("=" * 50)

# Get Original Boltz2 means as reference
original_means = {}
for energy_col, energy_title in energy_types.items():
    original_data = df[df['Run_Name'] == 'Original Boltz2'][energy_col]
    if len(original_data) > 0:
        original_means[energy_col] = original_data.mean()

# Calculate differences from original
differences_data = []
for energy_col, energy_title in energy_types.items():
    for run in run_order:
        if run != 'Original Boltz2':
            run_data = df[df['Run_Name'] == run][energy_col]
            if len(run_data) > 0 and energy_col in original_means:
                mean_diff = run_data.mean() - original_means[energy_col]
                differences_data.append({
                    'Energy_Type': energy_title,
                    'Run_Name': run,
                    'Difference_from_Original': mean_diff
                })

diff_df = pd.DataFrame(differences_data)

if not diff_df.empty:
    print("\nDifferences from Original Boltz2 (Positive = Less Favorable):")
    for energy_title in energy_types.values():
        print(f"\n{energy_title}:")
        energy_diff = diff_df[diff_df['Energy_Type'] == energy_title]
        for _, row in energy_diff.iterrows():
            direction = "worse" if row['Difference_from_Original'] > 0 else "better"
            print(f"  {row['Run_Name']:<25}: {row['Difference_from_Original']:+8.1f} kcal/mol ({direction})")

print("\n🧬 This analysis shows how much each perturbation affects binding energy!")